# RF Coach Vision: A/B dei pesi di TrackNet nel programma (badminton / run3 / run3b)

Analizza gli **stessi video** con **pesi di TrackNet diversi**, uno dopo l'altro, con il codice di oggi: così l'unica differenza tra i risultati sono i pesi. Ogni versione esce con il suo nome, accanto alle analisi di prima, che non vengono toccate:
- `<video>_badm`: pesi del programma (badminton, `TrackNet_best.pt`), rifatti con il codice di oggi;
- `<video>_run3`, `<video>_run3b`: pesi addestrati sui nostri video (`tracknet_finetune/run3`, `run3b`).

Su Drive vanno `outputs/dati/<video>_<pesi>_*` (CSV), `outputs/video/<video>_<pesi>_velocita.mp4` e `pred_result/`. Il confronto con i colpi veri lo fa Claude dai CSV.

**Video:** Nicola e Djokovic (`zverev_djokovic_trim_swin_like`), dove la velocità d'uscita funziona e ci sono i colpi veri, più alcaraz. Le versioni già su Drive (per esempio `alcaraz_run3`, fatta il 13/10 con lo stesso codice) vengono saltate (`SALTA_SE_FATTO`).

**Come si usa:** Runtime → Cambia tipo di runtime → **GPU T4** (o A100), poi **Runtime → Esegui tutto**. Serve solo l'autorizzazione di Drive (cella 2): le calibrazioni di questi video sono già su Drive, quindi non chiede niente. Durata indicativa: 5-10 minuti per analisi. Alla fine la sezione 7 mostra la tabella dei colpi per versione e la sezione 8 scollega il runtime (niente consumo a vuoto).

## 0. Video e pesi da confrontare

In [ ]:
# Video (in Drive/rf_coach_vision/inputs) e pesi da confrontare. Per aggiungere un video basta metterlo nella lista.
LISTA = ["nicola_matarese_trim.mp4", "zverev_djokovic_trim_swin_like.mp4", "alcaraz.mp4"]
DRIVE_DIR = "/content/drive/MyDrive/rf_coach_vision"
PESI_AB = {                       # etichetta -> file dei pesi ("" = pesi del programma, badminton)
    "badm": "",
    "run3": DRIVE_DIR + "/tracknet_finetune/run3/TrackNet_tennis.pt",
    "run3b": DRIVE_DIR + "/tracknet_finetune/run3b/TrackNet_tennis.pt",
}
SALTA_SE_FATTO = True             # salta le versioni che hanno gia' <video>_<etichetta>_velocita.csv su Drive
TRACKNET_MODE = "weight"          # come nel notebook principale

## 1. Controllo GPU

In [ ]:
!nvidia-smi -L

## 2. Collega Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Scarica il codice da GitHub

Il repository è **pubblico**: non serve nessun token.
Se un giorno tornasse **privato**, servirebbe un token: pannello a sinistra, icona della chiave (Secrets), un segreto chiamato `GITHUB_TOKEN` (token *fine-grained* con accesso al solo `rf_coach_vision` e permesso *Contents: Read-only*) con l'accesso al notebook attivo. La cella lo userà da sola.

In [ ]:
import os, shutil, glob, subprocess

REPO = "anrundo-2312/rf_coach_vision"
WORK_DIR = "/content/rf_coach_vision"

# Il repo e' pubblico: il token serve solo se un giorno torna privato.
token = ""
try:
    from google.colab import userdata
    token = (userdata.get("GITHUB_TOKEN") or "").strip()
except Exception:
    pass
url = f"https://{REPO.split('/')[0]}:{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

# Se la cella viene rieseguita, la cartella corrente e' quella che stiamo per
# cancellare (l'analisi fa %cd li'): git non partirebbe. Prima usciamo.
os.chdir("/content")
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
r = subprocess.run(["git", "clone", "--depth", "1", url, WORK_DIR], capture_output=True, text=True,
                   env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
if r.returncode != 0:
    raise SystemExit("Download da GitHub non riuscito:\n" + (r.stderr.replace(token, "***") if token else r.stderr))
print("codice scaricato:", sorted(os.listdir(WORK_DIR)))

## 4. Porta dal Drive modelli e calibrazioni, e prepara le copie dei video

Per ogni video e ogni versione si analizza una **copia** con il nome `<video>_<etichetta>.mp4`, con la sua calibrazione copiata da quella del video (come fa la cella 4b del notebook principale con `PESI`). Così i risultati delle tre versioni non si mescolano e quelli di prima restano.

In [ ]:
copie = [("models", ""), ("ckpts", "tracknet3/ckpts"), ("pred_result", "tracknet3/pred_result"),
         ("calibrazioni", "velocita/calibrazioni")]
for sub, dst in copie:
    s_dir = os.path.join(DRIVE_DIR, sub)
    d_dir = os.path.join(WORK_DIR, dst) if dst else WORK_DIR
    os.makedirs(d_dir, exist_ok=True)
    if os.path.isdir(s_dir):
        for f in glob.glob(os.path.join(s_dir, "*")):
            if os.path.isfile(f):
                shutil.copy(f, d_dir)

mancanti = [f for f in ["tennis_yolo11.pt", "yolo11n-pose.pt", "tracknet3/ckpts/TrackNet_best.pt"]
            if not os.path.exists(os.path.join(WORK_DIR, f))]
assert not mancanti, f"Mancano su Drive: {mancanti}"
for et, pesi in PESI_AB.items():
    assert not pesi or os.path.exists(pesi), f"pesi '{et}' non trovati: {pesi}"

os.makedirs(os.path.join(WORK_DIR, "inputs"), exist_ok=True)
DA_FARE = []                                    # (video originale, etichetta, nome della copia, pesi)
for v in LISTA:
    src_v = os.path.join(DRIVE_DIR, "inputs", v)
    assert os.path.exists(src_v), f"{v} non c'e' in Drive/rf_coach_vision/inputs"
    nome = v[:-4]
    cal = os.path.join(WORK_DIR, "velocita", "calibrazioni", nome + ".json")
    if not os.path.exists(cal):
        print(f"ATTENZIONE: {v} non ha la sua calibrazione: si usa quella standard")
    for et, pesi in PESI_AB.items():
        copia_nome = f"{nome}_{et}"
        fatto = os.path.exists(os.path.join(DRIVE_DIR, "outputs", "dati", copia_nome + "_velocita.csv"))
        if SALTA_SE_FATTO and fatto:
            print(f"{copia_nome}: gia' su Drive, salto")
            continue
        shutil.copy(src_v, os.path.join(WORK_DIR, "inputs", copia_nome + ".mp4"))
        if os.path.exists(cal):
            shutil.copy(cal, os.path.join(WORK_DIR, "velocita", "calibrazioni", copia_nome + ".json"))
        DA_FARE.append((v, et, copia_nome, pesi))
print("\nda analizzare:", [x[2] for x in DA_FARE])

## 5. Installa le librerie (PyTorch c'è già su Colab)

In [ ]:
!pip install -q ultralytics parse

## 6. Analisi, una versione dopo l'altra

Per ogni copia: analisi (`analyze.py` con i pesi della versione), classificazione dei colpi, velocità d'uscita, direzione, velocità dal rimbalzo e video con le velocità; poi salva subito su Drive i file di quella copia, così se la sessione cade quelle finite restano.

In [ ]:
import re, time
%cd {WORK_DIR}
ANNOTAZIONI = os.path.join(DRIVE_DIR, "datasets/tennis_actions/Tennis Player Actions Dataset for Human Pose Estimation/annotations")
if os.path.isdir(ANNOTAZIONI):
    !python classificazione/addestra_colpi.py --annotazioni "{ANNOTAZIONI}"
else:
    print("Annotazioni non trovate su Drive: uso il modello salvato nel repo.")

def imposta_analyze(pesi):
    p = os.path.join(WORK_DIR, "analyze.py")
    s = open(p).read()
    s = re.sub(r"^TRACKNET_MODE = .*$", f'TRACKNET_MODE = "{TRACKNET_MODE}"', s, flags=re.M)
    s = re.sub(r"^TRACKNET_FORCE_RECOMPUTE = .*$", "TRACKNET_FORCE_RECOMPUTE = False", s, flags=re.M)
    s = re.sub(r"^TRACKNET_PESI = .*$", f"TRACKNET_PESI = {pesi!r}", s, flags=re.M)
    open(p, "w").write(s)

t_tutto = time.time()
for v, et, NOME, pesi in DA_FARE:
    VIDEO = NOME + ".mp4"
    TRACKING = f"outputs/dati/{NOME}_tracking.csv"
    t0 = time.time()
    print(f"\n========== {NOME}  (pesi: {pesi or 'badminton'})")
    imposta_analyze(pesi)
    !python analyze.py inputs/{VIDEO}
    !python classificazione/classifica_tracking.py --tracking {TRACKING}
    !python classificazione/rileva_impatti.py --tracking {TRACKING}
    if os.path.exists(f"outputs/dati/{NOME}_impatti.csv"):
        !python classificazione/metriche_colpi.py --tracking {TRACKING}
    !python velocita/velocita_uscita.py --video inputs/{VIDEO}
    !python velocita/direzione_nascosta.py --video inputs/{VIDEO}
    !python velocita/velocita_rimbalzo.py --video inputs/{VIDEO}
    !python velocita/disegna_velocita.py --video inputs/{VIDEO}
    # su Drive: CSV, pallina e solo il video con le velocita' (gli altri video restano su Colab)
    for sub, s_dir, schema in [("outputs/dati", "outputs/dati", NOME + "_*"),
                               ("pred_result", "tracknet3/pred_result", NOME + "_*"),
                               ("outputs/video", "outputs/video", NOME + "_velocita.mp4")]:
        dst = os.path.join(DRIVE_DIR, sub)
        os.makedirs(dst, exist_ok=True)
        for f in glob.glob(os.path.join(WORK_DIR, s_dir, schema)):
            if os.path.isfile(f):
                shutil.copy(f, dst)
    print(f"{NOME}: {(time.time() - t0) / 60:.1f} minuti, salvato su Drive")
imposta_analyze("")                             # analyze.py torna come nel repo
print(f"\nTempo totale: {(time.time() - t_tutto) / 60:.1f} minuti")

## 7. Tabella dei colpi per versione

Per ogni video e ogni colpo trovato: velocità d'uscita, velocità dal rimbalzo, media fino al rimbalzo e direzione, nelle tre versioni. È solo una prima occhiata: il confronto con i colpi veri lo fa Claude dai CSV su Drive.

In [ ]:
import pandas as pd
from IPython.display import display
pd.set_option("display.width", 200)
for v in LISTA:
    righe = []
    for et in PESI_AB:
        p = os.path.join(DRIVE_DIR, "outputs", "dati", f"{v[:-4]}_{et}_velocita.csv")
        if not os.path.exists(p):
            print(f"{v[:-4]}_{et}: manca")
            continue
        d = pd.read_csv(p)
        for _, r in d.iterrows():
            righe.append({"versione": et, "frame": r["frame"], "colpo": r["colpo"],
                          "uscita": r.get("velocita_uscita_kmh"), "dal rimbalzo": r.get("velocita_rimbalzo_kmh"),
                          "media": r.get("velocita_media_kmh"), "direzione": r.get("direzione"),
                          "errore_px": r.get("errore_px")})
    print(f"\n===== {v}")
    if righe:
        display(pd.DataFrame(righe).sort_values(["frame", "versione"]).reset_index(drop=True))

## 8. Scollega il runtime

Tutto è già su Drive: scollegando si smette di consumare unità di calcolo. Se una cella prima si è fermata con un errore, questa non gira.

In [ ]:
print("Fatto: risultati su Drive in outputs/dati, outputs/video e pred_result.")
from google.colab import runtime
runtime.unassign()